# Clasificacion multiclase


Usa esta plantilla cuando la salida tenga más de dos clases.

Ejemplos: tipo de flor, categoría de cliente, diagnóstico entre varias enfermedades, clase A/B/C.

Aquí importa mirar clase por clase. Un modelo puede acertar mucho globalmente y fallar justo en la clase importante.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET = "target"  # CAMBIAR

df = pd.read_csv("data.csv")

print(df[TARGET].value_counts(dropna=False))

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))


## Cómo explicarlo

- La matriz de confusión muestra qué clases se confunden entre sí.
- `macro avg` trata todas las clases por igual, aunque una tenga pocas muestras.
- `weighted avg` pesa más las clases con muchas muestras.

Si hay desbalance, `macro avg` suele ser más honesta para ver si el modelo aprende todas las clases.


## Si falla

- Una clase tiene recall 0: probablemente hay desbalance o pocas muestras.
- Muchas confusiones entre dos clases: quizá las features no las separan bien.
- Train perfecto y test bajo: baja `max_depth` o sube `min_samples_leaf`.
- Si hay columnas de texto libre largas, esta plantilla no basta: habría que vectorizar texto.
